In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

import joblib
from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

In [ ]:
X_train = pd.read_csv("../data/X_train.csv")
X_test = pd.read_csv("../data/X_test.csv")
y_train = pd.read_csv("../data/y_train.csv").squeeze()
y_test = pd.read_csv("../data/y_test.csv").squeeze()


In [ ]:
# Originally used a fresh default logistic regression, with C of 1.0 and no elastic net, which was different from what model tuned and saved for deployment
# Loads saved pipeline instead now

BASE_DIR = Path("..")
model = joblib.load(BASE_DIR / "models" / "logistic_regression.pkl")
thresholds = joblib.load(BASE_DIR / "models" / "thresholds.pkl")
print(f"Loaded deployed model: C={thresholds['C']}, l1_ratio={thresholds['l1_ratio']}, threshold={thresholds['recall_threshold']}")

logistic_regression = model['model']
scalar = model['scaler']

X_train_scaled = pd.DataFrame(scalar.transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scalar.transform(X_test), columns=X_test.columns)




In [ ]:
masker = shap.maskers.Independent(X_train_scaled)
explainer = shap.LinearExplainer(logistic_regression, masker=masker)
shap_values = explainer.shap_values(X_train_scaled)
 
print(f"SHAP values shape: {shap_values.shape}")

In [ ]:
shap.summary_plot(shap_values, X_train_scaled, show=False)
plt.title("Feature impact on churn predicted value by SHAP value")
plt.tight_layout()
plt.show()

In [ ]:
shap.summary_plot(shap_values, X_train_scaled, plot_type="bar", show=False)
plt.title("Ranking of feature importance by mean SHAP value")
plt.tight_layout()
plt.show()

In [ ]:
feature_churn_importance = pd.DataFrame({
    "feature": X_train.columns,
    "mean_shap": np.abs(shap_values).mean(axis=0)
}).sort_values("mean_shap", ascending=False)
 
print("Top 10 features that drive churn:\n")
print(feature_churn_importance.head(10).to_string(index=False))
 
highest_three_features_by_churn_importance = feature_churn_importance["feature"].head(3).tolist()
 
for feature in highest_three_features_by_churn_importance:
    shap.dependence_plot(feature, shap_values, X_train_scaled, show=False)
    plt.title(f"SHAP Dependence - {feature}")
    plt.tight_layout()
    plt.show()

In [ ]:
churned_index = y_test[y_test == 1].index[0]
shap_values_test = explainer(X_test_scaled)
single_explanation = shap_values_test[churned_index]
shap.plots.waterfall(single_explanation, show=False)
plt.title("Biggest features that caused specific customer to churn")
plt.tight_layout()
plt.show()

In [ ]:
retained_index = y_test[y_test == 0].index[0]  
single_explanation = shap_values_test[retained_index]
shap.plots.waterfall(single_explanation, show=False)
plt.title("Biggest features that caused specific customer to stay")
plt.tight_layout()
plt.show()

In [ ]:
X_train.sample(200, random_state=42).to_csv("../models/shap_background.csv", index=False) # size of 200 randomnly chosen